<a href="https://colab.research.google.com/github/corebankingfiap-dev/core_banking/blob/main/04_01_RISCO_ESTATISTICO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Drawdown Histórico: Vamos calcular a queda do pico ao vale.
Value at Risk (VaR): Qual é o meu preujuízo?

Realizada leitura dos dados na camada gold. E salva os novos dados na pasta Reports dentro da camada god.

OBS:

GOLD/REPORTS/RISCO_BANCOS.parquet

GOLD/REPORTS/RISCO_COMMODITIES.parquet

GOLD/REPORTS/RISCO_FIIs.parquet


In [4]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Montar o Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. CONFIGURAÇÃO (Troque para "bancos" ou "commodities" quando precisar)
NOME_SEGMENTO = "commodities"

# Ajustado para o nome que aparece na sua foto da Silver (é mais preciso para risco diário)
ARQUIVO = f"b3_{NOME_SEGMENTO}_limpo.csv"
BASE_PATH_SILVER = '/content/drive/MyDrive/DATA_LAKE/02_Silver/'
BASE_PATH_GOLD = '/content/drive/MyDrive/DATA_LAKE/03_Gold/'

full_path = os.path.join(BASE_PATH_SILVER, ARQUIVO)

if os.path.exists(full_path):
    # Lendo os preços limpos da Silver
    df_precos = pd.read_csv(full_path, index_col=0, parse_dates=True)

    # Transformando preços em retornos diários para o cálculo de risco
    df_retornos = df_precos.pct_change().dropna()

    # 3. FUNÇÃO DE CÁLCULO DE RISCO
    def calc_risk_metrics(s):
        # Max Drawdown
        wealth = (1 + s).cumprod()
        peak = wealth.cummax()
        dd = (wealth - peak) / peak

        # VaR 95% e CVaR (Expected Shortfall)
        v95 = s.quantile(0.05)
        c95 = s[s <= v95].mean()

        return pd.Series({
            'Max_Drawdown': dd.min(),
            'VaR_95_Diario': v95,
            'CVaR_95_Diario': c95
        })

    # 4. PROCESSAMENTO
    relatorio = df_retornos.apply(calc_risk_metrics).T

    # 5. FORMATAÇÃO PARA EXIBIÇÃO (Bonitinha com %)
    relatorio_view = relatorio.copy()
    for col in relatorio_view.columns:
        relatorio_view[col] = relatorio_view[col].map(lambda x: f"{x:.2%}")

    # 6. SALVAMENTO NA GOLD (Dentro de uma pasta REPORTS)
    path_reports = os.path.join(BASE_PATH_GOLD, 'REPORTS')
    os.makedirs(path_reports, exist_ok=True)

    # Salvando os resultados
    relatorio.to_csv(os.path.join(path_reports, f"RISCO_{NOME_SEGMENTO.upper()}.csv"))
    relatorio_view.to_csv(os.path.join(path_reports, f"RELATORIO_FINAL_{NOME_SEGMENTO.upper()}.csv"))

    print(f"\n✅ RELATÓRIO DE RISCO {NOME_SEGMENTO.upper()} GERADO")
    print("-" * 60)
    print(relatorio_view)
    print("-" * 60)
    print(f"Arquivos salvos em: {path_reports}")

else:
    print(f"❌ Arquivo não encontrado! Verifique se o nome é {ARQUIVO} na pasta Silver.")


✅ RELATÓRIO DE RISCO COMMODITIES GERADO
------------------------------------------------------------
      Max_Drawdown VaR_95_Diario CVaR_95_Diario
CSNA3      -81.98%        -4.78%         -6.19%
PETR4      -39.10%        -3.00%         -4.65%
PRIO3      -35.33%        -3.69%         -5.32%
SUZB3      -47.35%        -2.68%         -3.70%
VALE3      -41.34%        -2.91%         -3.87%
------------------------------------------------------------
Arquivos salvos em: /content/drive/MyDrive/DATA_LAKE/03_Gold/REPORTS


Max_Drawdown (-50.49%): Indica uma perda de valor em relação ao pico. Se o sinal fosse positivo, pareceria que o ativo subiu 50% em vez de cair. É um indicador de queda acumulada.

VaR e CVaR (-11.56%): Representam uma saída de capital ou desvalorização esperada. O sinal negativo reforça que estamos falando de um prejuízo potencial, não de um lucro.